# starfold applied to APOGEE DR17 x astroNN

This optional tutorial shows `starfold`'s methodology applied to the same
kind of data the paper used: the APOGEE DR17 x astroNN value-added catalogue
(Leung et al. 2023), which provides ages, abundances, and galactocentric
kinematics for ~700 000 APOGEE stars.

The feature choice below (age, [Fe/H], [Mg/Fe], $V_\phi$, $\sqrt{V_R^2 + V_z^2}$)
is the *user's* choice to illustrate the tool on a chrono-chemo-kinematic space.
It is not baked into `starfold`; a different domain would pick different columns.

## One-time data setup

This notebook does **not** download the full catalogue automatically -- it is
~650 MB and the exact subset a user wants is application-dependent. Download
it once manually:

```bash
curl -L -o apogee_astroNN-DR17.fits \
    https://data.sdss.org/sas/dr17/env/APOGEE_ASTRO_NN/apogee_astroNN-DR17.fits
```

and set the environment variable `STARFOLD_ASTRONN_FITS` to point at it, or
drop the file at `~/data/apogee_astroNN-DR17.fits`. If the file is absent,
the notebook prints a hint and stops cleanly.

Required optional packages (not installed by `starfold` itself):

```bash
pip install astropy
```

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import starfold as sf

## 1. Locate the catalogue

We look in `$STARFOLD_ASTRONN_FITS`, then `~/data/`, then the working directory.
If nothing is found, the notebook exits with a hint.

In [ ]:
CANDIDATES = [
    os.environ.get("STARFOLD_ASTRONN_FITS"),
    Path.home() / "data" / "apogee_astroNN-DR17.fits",
    Path.cwd() / "apogee_astroNN-DR17.fits",
]
fits_path: Path | None = None
for cand in CANDIDATES:
    if cand and Path(cand).exists():
        fits_path = Path(cand)
        break
if fits_path is None:
    msg = (
        "apogee_astroNN-DR17.fits not found. Download it once (see intro cell) "
        "and set STARFOLD_ASTRONN_FITS or drop it in ~/data/."
    )
    raise SystemExit(msg)
print(f"Using {fits_path} ({fits_path.stat().st_size / 1e6:.0f} MB)")

## 2. Load, filter, deduplicate

We pull only the columns we need, apply the paper's quality cuts (finite
values, `age_lowess_correct` in a physically meaningful range, no extreme
kinematic outliers), and deduplicate on `APOGEE_ID` keeping the highest-SNR
visit per star.

In [ ]:
from astropy.io import fits  # type: ignore[import-untyped]

COLS = ("APOGEE_ID", "SNR", "FE_H", "MG_FE", "age_lowess_correct", "galvt", "galvR", "galvz")
with fits.open(fits_path, memmap=True) as hdul:
    data = hdul[1].data  # noqa: PD002 -- astropy Table, not pandas
    cols = {c: np.asarray(data[c]) for c in COLS}

valid = np.ones(cols["APOGEE_ID"].shape[0], dtype=bool)
for c in ("FE_H", "MG_FE", "age_lowess_correct", "galvt", "galvR", "galvz", "SNR"):
    valid &= np.isfinite(cols[c])
valid &= (cols["age_lowess_correct"] > 0) & (cols["age_lowess_correct"] < 14)
valid &= cols["SNR"] > 70
for c in ("galvt", "galvR", "galvz"):
    valid &= np.abs(cols[c]) < 500  # km/s; trim obvious kinematic outliers
for c in COLS:
    cols[c] = cols[c][valid]
print(f"{valid.sum():_} rows after quality cuts")

# Deduplicate on APOGEE_ID keeping highest SNR.
order = np.argsort(cols["SNR"])[::-1]
_, first_idx = np.unique(cols["APOGEE_ID"][order], return_index=True)
keep = np.sort(order[first_idx])
for c in COLS:
    cols[c] = cols[c][keep]
print(f"{keep.size:_} unique stars")

## 3. Assemble the feature matrix

Five features:
- `age` (Gyr)
- `[Fe/H]`
- `[Mg/Fe]`
- `galvt` -- azimuthal velocity $V_\phi$ (km/s)
- `V_perp = sqrt(galvR^2 + galvz^2)` (km/s)

`starfold` takes any numeric `(n_samples, n_features)` array, so the feature
choice is the caller's responsibility. To keep the demo tractable we draw a
random 20 000-star subsample; remove the cap if you have time for a full run.

In [ ]:
rng = np.random.default_rng(0)
n_full = cols["FE_H"].shape[0]
n_demo = min(20_000, n_full)
idx = rng.choice(n_full, size=n_demo, replace=False)

v_perp = np.sqrt(cols["galvR"][idx] ** 2 + cols["galvz"][idx] ** 2)
X = np.column_stack([
    cols["age_lowess_correct"][idx],
    cols["FE_H"][idx],
    cols["MG_FE"][idx],
    cols["galvt"][idx],
    v_perp,
])
print("X shape:", X.shape)

## 4. Run the pipeline

On 20 000 stars with the paper-ish defaults this takes a few minutes. The
noise baseline is skipped here for speed; set `skip_noise_baseline=False` and
bump `n_realisations` in production.

In [ ]:
pipeline = sf.UnsupervisedPipeline(
    umap_kwargs={"n_neighbors": 15, "min_dist": 0.0, "n_epochs": 2000},
    hdbscan_optuna_trials=60,
    mcs_range=(50, 500),
    ms_range=(5, 50),
    skip_noise_baseline=True,
    random_state=0,
)
result = pipeline.fit(X)
print(result.summary())

## 5. Diagnostic plots

The embedding and the Optuna search history.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sf.plot_embedding(result.embedding, result.labels, ax=axes[0])
axes[0].set_title(f"{result.n_clusters} clusters (T = {result.trustworthiness:.3f})")
sf.plot_optuna_history(result.search.study, ax=axes[1])
plt.tight_layout()
plt.show()

## 6. Inspect clusters in the original feature space

Clusters found by HDBSCAN in the embedding have well-defined positions in the
original physical features. Below, median values per cluster -- the kind of
summary you would cross-check against domain knowledge before trusting the
decomposition.

In [ ]:
feature_names = ["age (Gyr)", "[Fe/H]", "[Mg/Fe]", "Vphi (km/s)", "Vperp (km/s)"]
for label in np.unique(result.labels):
    mask = result.labels == label
    medians = np.median(X[mask], axis=0)
    tag = "outliers" if label == -1 else f"cluster {label}"
    parts = [f"{n} = {v:+6.2f}" for n, v in zip(feature_names, medians, strict=True)]
    print(f"{tag:>12s}  (n = {mask.sum():_})  {'  '.join(parts)}")

## 7. Caveats

- This notebook stops at the methodology. It does not attempt to identify
  clusters as "thick disk", "halo", or similar -- that is a scientific-context
  step outside the scope of this package.
- A ~20 k subsample is for demonstration. The paper clusters ~130 k stars;
  remove the `n_demo` cap and expect hours, not minutes.
- The two-run workflow (re-cluster within each top-level component) is
  illustrated in `tutorial_01_quickstart.ipynb`; apply it here by filtering
  `X[result.labels == k]` and calling `pipeline.fit` again.